# WorldSim v2 Training Data Generation

Generate Tasks I–N + Task G Korean fix via Sonnet API (OpenRouter).


## 1. Repo Root & Environment


In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "config" / "generation.yaml").exists():
        raise RuntimeError("Cannot find repo root. Run this notebook from the worldsim-training directory.")

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from scripts.generate_data import load_local_env
load_local_env(REPO_ROOT)

api_key = os.environ.get("OPENROUTER_API_KEY", "")
model = os.environ.get("OPENROUTER_MODEL", "")
print(f"Repo root: {REPO_ROOT}")
print(f"API key:   {'✅ set (' + api_key[:12] + '...)' if api_key else '❌ MISSING — set OPENROUTER_API_KEY in .env'}")
print(f"Model:     {model or '⚠️ using default from generation.yaml'}")
assert api_key, "OPENROUTER_API_KEY is required. Add it to .env file."


## 2. Batch Preview (Dry Run)


In [ ]:
from collections import Counter

from scripts.generate_data import (
    apply_batch_plan_to_settings,
    batch_task_counts,
    build_jobs,
    load_batch_plan,
    load_generation_config,
    select_requested_jobs,
)

for batch_id in ["batch_v2_01_tasks_in", "batch_v2_02_task_g_fix"]:
    batch_plan = load_batch_plan(REPO_ROOT, batch_id=batch_id)
    settings = apply_batch_plan_to_settings(load_generation_config(REPO_ROOT / "config"), batch_plan)
    jobs = build_jobs(REPO_ROOT, settings_override=settings)
    selected = select_requested_jobs(jobs, task_counts=batch_task_counts(batch_plan))
    counts = Counter(j["task"] for j in selected)
    print(f"\n{'=' * 40}")
    print(f"Batch: {batch_id}")
    print(f"Total jobs: {len(selected)}")
    for task, count in sorted(counts.items()):
        print(f"  Task {task}: {count}")
    print(f"{'=' * 40}")


## 3. Batch 1: Tasks I–N Generation


In [ ]:
import time

from scripts.generate_data import generate_dataset, load_batch_plan

batch_plan_1 = load_batch_plan(REPO_ROOT, batch_id="batch_v2_01_tasks_in")
print(f"Starting Batch 1: Tasks I–N ({sum(batch_plan_1.get('task_counts', {}).values())} examples)")
print("Estimated cost: ~$4.55")
print("Estimated time: 30-60 minutes")
print("=" * 50)

started = time.perf_counter()
result_1 = generate_dataset(REPO_ROOT, batch_plan=batch_plan_1)
elapsed_1 = time.perf_counter() - started

print(f"\n{'=' * 50}")
print(f"Batch 1 complete in {elapsed_1:.0f}s ({elapsed_1 / 60:.1f}min)")
result_1


## 4. Batch 1 Summary


In [ ]:
import json

batch1_dir = REPO_ROOT / "data" / "raw" / "batch_v2_01_tasks_in"
summary_path = batch1_dir / "summary.json"
generated_path = batch1_dir / "generated.jsonl"
skipped_path = batch1_dir / "skipped.jsonl"

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("=== Batch 1 Summary ===")
    for key in ["total_generated", "total_skipped", "total_cost_usd", "elapsed_seconds"]:
        print(f"  {key}: {summary.get(key, 'N/A')}")
    print("\n  Task breakdown:")
    for task, count in sorted(summary.get("task_counts", {}).items()):
        print(f"    Task {task}: {count}")

generated_count = sum(1 for _ in open(generated_path, encoding="utf-8")) if generated_path.exists() else 0
skipped_count = sum(1 for _ in open(skipped_path, encoding="utf-8")) if skipped_path.exists() else 0
print(f"\n  Generated rows: {generated_count}")
print(f"  Skipped rows:  {skipped_count}")

if generated_path.exists():
    print("\n=== Sample Outputs ===")
    with open(generated_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            row = json.loads(line)
            print(f"\n--- Task {row.get('task')} ---")
            output = row.get("output", row.get("assistant", ""))
            if isinstance(output, str) and len(output) > 200:
                output = output[:200] + "..."
            print(f"  Output: {output}")


## 5. Batch 2: Task G Korean Fix


In [ ]:
batch_plan_2 = load_batch_plan(REPO_ROOT, batch_id="batch_v2_02_task_g_fix")
print(f"Starting Batch 2: Task G Korean fix ({sum(batch_plan_2.get('task_counts', {}).values())} examples)")
print("Estimated cost: ~$0.90")
print("=" * 50)

started = time.perf_counter()
result_2 = generate_dataset(REPO_ROOT, batch_plan=batch_plan_2)
elapsed_2 = time.perf_counter() - started

print(f"\n{'=' * 50}")
print(f"Batch 2 complete in {elapsed_2:.0f}s ({elapsed_2 / 60:.1f}min)")
result_2


## 6. Batch 2 Summary


In [ ]:
batch2_dir = REPO_ROOT / "data" / "raw" / "batch_v2_02_task_g_fix"
summary_path_2 = batch2_dir / "summary.json"
generated_path_2 = batch2_dir / "generated.jsonl"

if summary_path_2.exists():
    summary_2 = json.loads(summary_path_2.read_text(encoding="utf-8"))
    print("=== Batch 2 Summary ===")
    for key in ["total_generated", "total_skipped", "total_cost_usd", "elapsed_seconds"]:
        print(f"  {key}: {summary_2.get(key, 'N/A')}")

generated_count_2 = sum(1 for _ in open(generated_path_2, encoding="utf-8")) if generated_path_2.exists() else 0
print(f"\n  Generated rows: {generated_count_2}")

if generated_path_2.exists():
    print("\n=== Task G Sample Outputs (check Korean quality) ===")
    with open(generated_path_2, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            row = json.loads(line)
            output = row.get("output", row.get("assistant", ""))
            print(f"\n--- G sample {i + 1} ---")
            if isinstance(output, str):
                try:
                    parsed = json.loads(output)
                    print(f"  interpretation_ko: {parsed.get('interpretation_ko', 'N/A')}")
                    print(f"  interpretation_en: {parsed.get('interpretation_en', 'N/A')}")
                except json.JSONDecodeError:
                    print(f"  Raw: {output[:200]}")


## 7. Negative Examples & General Korean Stats


In [ ]:
from collections import Counter

neg_path = REPO_ROOT / "data" / "samples" / "negative_examples.jsonl"
gen_path = REPO_ROOT / "data" / "samples" / "general_korean.jsonl"

for label, path in [("Negative Examples", neg_path), ("General Korean", gen_path)]:
    if path.exists():
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        cats = Counter(r.get("reason", r.get("category", "unknown")) for r in rows)
        print(f"\n=== {label}: {len(rows)} total ===")
        for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
            print(f"  {cat}: {count}")
    else:
        print(f"\n⚠️ {label}: file not found at {path}")


## 8. Validation


In [ ]:
from scripts.common import load_yaml, resolve_path
from scripts.validate_data import _load_batch_plan as load_validation_batch_plan
from scripts.validate_data import _resolve_batch_paths, load_validation_rules, validate_file

print("=== Validating Batch 1 (Tasks I–N) ===")
batch1_generated = REPO_ROOT / "data" / "raw" / "batch_v2_01_tasks_in" / "generated.jsonl"
if batch1_generated.exists():
    settings = load_yaml(REPO_ROOT / "config" / "generation.yaml")
    batch_plan = load_validation_batch_plan(REPO_ROOT, "batch_v2_01_tasks_in")
    input_path, output_dir = _resolve_batch_paths(REPO_ROOT, settings, batch_plan)
    summary = validate_file(input_path=input_path, validated_dir=output_dir, rules=load_validation_rules(REPO_ROOT / "config"))
    print(summary)
else:
    print("  ⚠️ Batch 1 not yet generated")

print("\n=== Validating Batch 2 (Task G fix) ===")
batch2_generated = REPO_ROOT / "data" / "raw" / "batch_v2_02_task_g_fix" / "generated.jsonl"
if batch2_generated.exists():
    settings = load_yaml(REPO_ROOT / "config" / "generation.yaml")
    batch_plan = load_validation_batch_plan(REPO_ROOT, "batch_v2_02_task_g_fix")
    input_path, output_dir = _resolve_batch_paths(REPO_ROOT, settings, batch_plan)
    summary = validate_file(input_path=input_path, validated_dir=output_dir, rules=load_validation_rules(REPO_ROOT / "config"))
    print(summary)
else:
    print("  ⚠️ Batch 2 not yet generated")


## 9. Overall v2 Data Status


In [ ]:
print("=" * 60)
print("V2 DATA GENERATION STATUS")
print("=" * 60)

sources = {
    "v1 train": REPO_ROOT / "data" / "training" / "worldsim-v31-mix-v1" / "train_converted.jsonl",
    "v1 dev": REPO_ROOT / "data" / "training" / "worldsim-v31-mix-v1" / "dev_converted.jsonl",
    "Batch 1 (I-N)": REPO_ROOT / "data" / "raw" / "batch_v2_01_tasks_in" / "generated.jsonl",
    "Batch 2 (G fix)": REPO_ROOT / "data" / "raw" / "batch_v2_02_task_g_fix" / "generated.jsonl",
    "Negative": REPO_ROOT / "data" / "samples" / "negative_examples.jsonl",
    "General KR": REPO_ROOT / "data" / "samples" / "general_korean.jsonl",
}

grand_total = 0
for label, path in sources.items():
    if path.exists():
        count = sum(1 for line in open(path, encoding="utf-8") if line.strip())
        print(f"  {label:20s}: {count:>5} rows")
        grand_total += count
    else:
        print(f"  {label:20s}:   N/A (not yet generated)")

print(f"\n  {'GRAND TOTAL':20s}: {grand_total:>5} rows")
print(f"\n  Estimated API cost: ~${4.55 + 0.90:.2f}")
print("=" * 60)
